# BERTweet Domain Fine-Tuning on Colab GPU

Use this notebook in Google Colab with **Runtime > Change runtime type > GPU**. It installs the repo dependencies, imports the configured Hugging Face datasets, trains B0, and writes the comparison report.

In [1]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

CUDA available: False
/bin/bash: line 1: nvidia-smi: command not found


## Get the repo

Set `REPO_URL` after pushing this repository to GitHub. If the repo is already available under `/content/bertweet-domain-ft-experiment`, leave it empty.

In [ ]:
from pathlib import Path
import os

REPO_URL = ''  # Example: 'https://github.com/your-name/bertweet-domain-ft-experiment.git'
PROJECT_DIR = Path('/content/bertweet-domain-ft-experiment')

if REPO_URL and not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}
elif not PROJECT_DIR.exists():
    raise RuntimeError('Set REPO_URL or upload/clone the repo to /content/bertweet-domain-ft-experiment first.')

os.chdir(PROJECT_DIR)
print('Working directory:', Path.cwd())

In [ ]:
!python -m pip install -U pip
!pip install -r requirements.txt

In [ ]:
!python -m pytest -q
!python -m compileall src

In [ ]:
!python -m src.hf_data --config configs/experiment_colab.yaml --overwrite
!python -m src.run_baseline --config configs/experiment_colab.yaml
!python -m src.compare --config configs/experiment_colab.yaml
!python -m src.report --config configs/experiment_colab.yaml

In [ ]:
import pandas as pd
pd.read_csv('outputs_colab/tables/baseline_metrics.csv')[['eval_set', 'accuracy', 'f1_macro', 'training_time_seconds', 'peak_gpu_memory_mb']]

## Optional: run the full treatment matrix

The full matrix runs Full FT, LoRA ranks 8/16/32, and Adapter for both Twitter and general domains. It can take longer and use more GPU memory than B0.

In [ ]:
# !python -m src.run_matrix --config configs/experiment_colab.yaml
# !python -m src.compare --config configs/experiment_colab.yaml
# !python -m src.report --config configs/experiment_colab.yaml